In [1]:
# -----------------------------
# Imports
# -----------------------------
from pathlib import Path
import re
import pandas as pd
import numpy as np

# Display settings for easier notebook inspection
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", None)

# -----------------------------
# Paths
# -----------------------------
REPO_ROOT = Path("..").resolve()
DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"

# Change this to the actual filename of the large Kaggle CSV
DEMO_JOBS_FILE = "job_descriptions.csv"

print("REPO_ROOT:", REPO_ROOT)
print("DATA_RAW exists:", DATA_RAW.exists())
print("DATA_PROCESSED exists:", DATA_PROCESSED.exists())
print("Raw file path:", DATA_RAW / DEMO_JOBS_FILE)

REPO_ROOT: C:\Users\COMPUTER CARE\aeej1\JobPlatform
DATA_RAW exists: True
DATA_PROCESSED exists: True
Raw file path: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\raw\job_descriptions.csv


In [2]:
# -----------------------------
# Read only the header first to inspect the actual column names.
# -----------------------------
raw_columns = pd.read_csv(DATA_RAW / DEMO_JOBS_FILE, nrows=0).columns.tolist()

print("Raw columns found in CSV:")
print(raw_columns)

Raw columns found in CSV:
['Job Id', 'Experience', 'Qualifications', 'Salary Range', 'location', 'Country', 'latitude', 'longitude', 'Work Type', 'Company Size', 'Job Posting Date', 'Preference', 'Contact Person', 'Contact', 'Job Title', 'Role', 'Job Portal', 'Job Description', 'Benefits', 'skills', 'Responsibilities', 'Company', 'Company Profile']


In [3]:
# -----------------------------
# Build a mapping from the actual CSV column names to our internal names.
#
# Important:
# Kaggle datasets sometimes vary in capitalization, spacing,
# or exact naming (e.g. Company vs Company Name, location vs Location).
# So we match against the actual headers found in Cell 1.
# -----------------------------
COLUMN_MAPPING = {}

for col in raw_columns:
    c = col.strip().lower()

    if c == "job id":
        COLUMN_MAPPING[col] = "job_id"
    elif c == "experience":
        COLUMN_MAPPING[col] = "experience"
    elif c == "qualifications":
        COLUMN_MAPPING[col] = "qualifications"
    elif c == "location":
        COLUMN_MAPPING[col] = "location"
    elif c == "country":
        COLUMN_MAPPING[col] = "country"
    elif c == "work type":
        COLUMN_MAPPING[col] = "work_type"
    elif c == "job posting date":
        COLUMN_MAPPING[col] = "job_posting_date"
    elif c == "job title":
        COLUMN_MAPPING[col] = "job_title"
    elif c == "role":
        COLUMN_MAPPING[col] = "role"
    elif c == "job portal":
        COLUMN_MAPPING[col] = "job_portal"
    elif c == "job description":
        COLUMN_MAPPING[col] = "job_description"
    elif c == "benefits":
        COLUMN_MAPPING[col] = "benefits"
    elif c == "skills":
        COLUMN_MAPPING[col] = "skills"
    elif c == "responsibilities":
        COLUMN_MAPPING[col] = "responsibilities"
    elif c in ["company name", "company"]:
        COLUMN_MAPPING[col] = "company_name"
    elif c == "company profile":
        COLUMN_MAPPING[col] = "company_profile"

print("Detected mapping:")
print(COLUMN_MAPPING)

Detected mapping:
{'Job Id': 'job_id', 'Experience': 'experience', 'Qualifications': 'qualifications', 'location': 'location', 'Country': 'country', 'Work Type': 'work_type', 'Job Posting Date': 'job_posting_date', 'Job Title': 'job_title', 'Role': 'role', 'Job Portal': 'job_portal', 'Job Description': 'job_description', 'Benefits': 'benefits', 'skills': 'skills', 'Responsibilities': 'responsibilities', 'Company': 'company_name', 'Company Profile': 'company_profile'}


In [4]:
# -----------------------------
# Load only the columns we successfully mapped.
# This keeps memory usage lower than loading the full 1.75 GB file blindly.
# -----------------------------
use_columns = list(COLUMN_MAPPING.keys())

demo_jobs_raw = pd.read_csv(
    DATA_RAW / DEMO_JOBS_FILE,
    usecols=use_columns
)

print("Loaded shape:", demo_jobs_raw.shape)
print("\nLoaded columns:")
print(demo_jobs_raw.columns.tolist())

display(demo_jobs_raw.head(3))

Loaded shape: (1615940, 16)

Loaded columns:
['Job Id', 'Experience', 'Qualifications', 'location', 'Country', 'Work Type', 'Job Posting Date', 'Job Title', 'Role', 'Job Portal', 'Job Description', 'Benefits', 'skills', 'Responsibilities', 'Company', 'Company Profile']


,Job Id,Experience,Qualifications,location,Country,Work Type,Job Posting Date,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile
0,1089843540111562,5 to 15 Years,M.Tech,Douglas,Isle of Man,Intern,2022-04-24,Digital Marketing Specialist,Social Media Manager,Snagajob,"Social Media Managers oversee an organizations social media presence. They create and schedule content, engage with followers, and analyze social media metrics to drive brand awareness and engagement.","{'Flexible Spending Accounts (FSAs), Relocation Assistance, Legal Assistance, Employee Recognition Programs, Financial Counseling'}","Social media platforms (e.g., Facebook, Twitter, Instagram) Content creation and scheduling Social media analytics and insights Community engagement Paid social advertising","Manage and grow social media accounts, create engaging content, and interact with the online community. Develop social media content calendars and strategies. Monitor social media trends and engagement metrics.",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversified Financials"",""City"":""Sunny Isles Beach"",""State"":""Florida"",""Zip"":""33160"",""Website"":""www.ielp.com"",""Ticker"":""IEP"",""CEO"":""David Willetts""}"
1,398454096642776,2 to 12 Years,BCA,Ashgabat,Turkmenistan,Intern,2022-12-19,Web Developer,Frontend Web Developer,Idealist,"Frontend Web Developers design and implement user interfaces for websites, ensuring they are visually appealing and user-friendly. They collaborate with designers and backend developers to create seamless web experiences for users.","{'Health Insurance, Retirement Plans, Paid Time Off (PTO), Flexible Work Arrangements, Employee Assistance Programs (EAP)'}","HTML, CSS, JavaScript Frontend frameworks (e.g., React, Angular) User experience (UX)","Design and code user interfaces for websites, ensuring a seamless and visually appealing user experience. Collaborate with UX designers to optimize user journeys. Ensure cross-browser compatibility and responsive design.",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Commercial Banks"",""City"":""Pittsburgh"",""State"":""Pennsylvania"",""Zip"":""15222"",""Website"":""www.pnc.com"",""Ticker"":""PNC"",""CEO"":""William S. Demchak""}"
2,481640072963533,0 to 12 Years,PhD,Macao,"Macao SAR, China",Temporary,2022-09-14,Operations Manager,Quality Control Manager,Jobs2Careers,"Quality Control Managers establish and enforce quality standards within an organization. They develop quality control processes, perform inspections, and implement corrective actions to maintain product or service quality.","{'Legal Assistance, Bonuses and Incentive Programs, Wellness Programs, Employee Discounts, Retirement Plans'}","Quality control processes and methodologies Statistical process control (SPC) Root cause analysis and corrective action Quality management systems (e.g., ISO 9001) Compliance and regulatory knowledge",Establish and enforce quality control standards and procedures. Conduct quality audits and inspections. Collaborate with production teams to address quality issues and implement improvements.,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: Property and Casualty (Stock)"",""City"":""San Antonio"",""State"":""Texas"",""Zip"":""78288"",""Website"":""www.usaa.com"",""Ticker"":"""",""CEO"":""Wayne Peacock""}"


In [5]:
# -----------------------------
# Rename the loaded columns into our stable snake_case schema.
# -----------------------------
demo_jobs = demo_jobs_raw.rename(columns=COLUMN_MAPPING).copy()

print("Columns after rename:")
print(demo_jobs.columns.tolist())

display(demo_jobs.head(3))

Columns after rename:
['job_id', 'experience', 'qualifications', 'location', 'country', 'work_type', 'job_posting_date', 'job_title', 'role', 'job_portal', 'job_description', 'benefits', 'skills', 'responsibilities', 'company_name', 'company_profile']


,job_id,experience,qualifications,location,country,work_type,job_posting_date,job_title,role,job_portal,job_description,benefits,skills,responsibilities,company_name,company_profile
0,1089843540111562,5 to 15 Years,M.Tech,Douglas,Isle of Man,Intern,2022-04-24,Digital Marketing Specialist,Social Media Manager,Snagajob,"Social Media Managers oversee an organizations social media presence. They create and schedule content, engage with followers, and analyze social media metrics to drive brand awareness and engagement.","{'Flexible Spending Accounts (FSAs), Relocation Assistance, Legal Assistance, Employee Recognition Programs, Financial Counseling'}","Social media platforms (e.g., Facebook, Twitter, Instagram) Content creation and scheduling Social media analytics and insights Community engagement Paid social advertising","Manage and grow social media accounts, create engaging content, and interact with the online community. Develop social media content calendars and strategies. Monitor social media trends and engagement metrics.",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversified Financials"",""City"":""Sunny Isles Beach"",""State"":""Florida"",""Zip"":""33160"",""Website"":""www.ielp.com"",""Ticker"":""IEP"",""CEO"":""David Willetts""}"
1,398454096642776,2 to 12 Years,BCA,Ashgabat,Turkmenistan,Intern,2022-12-19,Web Developer,Frontend Web Developer,Idealist,"Frontend Web Developers design and implement user interfaces for websites, ensuring they are visually appealing and user-friendly. They collaborate with designers and backend developers to create seamless web experiences for users.","{'Health Insurance, Retirement Plans, Paid Time Off (PTO), Flexible Work Arrangements, Employee Assistance Programs (EAP)'}","HTML, CSS, JavaScript Frontend frameworks (e.g., React, Angular) User experience (UX)","Design and code user interfaces for websites, ensuring a seamless and visually appealing user experience. Collaborate with UX designers to optimize user journeys. Ensure cross-browser compatibility and responsive design.",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Commercial Banks"",""City"":""Pittsburgh"",""State"":""Pennsylvania"",""Zip"":""15222"",""Website"":""www.pnc.com"",""Ticker"":""PNC"",""CEO"":""William S. Demchak""}"
2,481640072963533,0 to 12 Years,PhD,Macao,"Macao SAR, China",Temporary,2022-09-14,Operations Manager,Quality Control Manager,Jobs2Careers,"Quality Control Managers establish and enforce quality standards within an organization. They develop quality control processes, perform inspections, and implement corrective actions to maintain product or service quality.","{'Legal Assistance, Bonuses and Incentive Programs, Wellness Programs, Employee Discounts, Retirement Plans'}","Quality control processes and methodologies Statistical process control (SPC) Root cause analysis and corrective action Quality management systems (e.g., ISO 9001) Compliance and regulatory knowledge",Establish and enforce quality control standards and procedures. Conduct quality audits and inspections. Collaborate with production teams to address quality issues and implement improvements.,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: Property and Casualty (Stock)"",""City"":""San Antonio"",""State"":""Texas"",""Zip"":""78288"",""Website"":""www.usaa.com"",""Ticker"":"""",""CEO"":""Wayne Peacock""}"


In [6]:
# -----------------------------
# Some dataset versions may not include every desired field.
# To keep the pipeline stable, created missing columns as empty strings.
# -----------------------------
REQUIRED_COLUMNS = [
    "job_id",
    "experience",
    "qualifications",
    "location",
    "country",
    "work_type",
    "job_posting_date",
    "job_title",
    "role",
    "job_portal",
    "job_description",
    "benefits",
    "skills",
    "responsibilities",
    "company_name",
    "company_profile",
]

for col in REQUIRED_COLUMNS:
    if col not in demo_jobs.columns:
        demo_jobs[col] = ""

# Fallback:
# If role is missing or blank, use job_title.
demo_jobs["role"] = demo_jobs["role"].replace("", np.nan)
demo_jobs["role"] = demo_jobs["role"].fillna(demo_jobs["job_title"])

print("Columns after ensuring required schema:")
print(demo_jobs.columns.tolist())

display(demo_jobs[["job_title", "role"]].head(5))

Columns after ensuring required schema:
['job_id', 'experience', 'qualifications', 'location', 'country', 'work_type', 'job_posting_date', 'job_title', 'role', 'job_portal', 'job_description', 'benefits', 'skills', 'responsibilities', 'company_name', 'company_profile']


,job_title,role
0,Digital Marketing Specialist,Social Media Manager
1,Web Developer,Frontend Web Developer
2,Operations Manager,Quality Control Manager
3,Network Engineer,Wireless Network Engineer
4,Event Manager,Conference Manager


In [7]:
# -----------------------------
# Helper to safely clean text
# -----------------------------
def clean_text(x):
    if pd.isna(x):
        return ""

    s = str(x)
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()

In [8]:
# -----------------------------
# Clean the important text columns so later filtering
# and deduplication are more reliable.
# -----------------------------
TEXT_COLUMNS = [
    "experience",
    "qualifications",
    "location",
    "country",
    "work_type",
    "job_posting_date",
    "job_title",
    "role",
    "job_portal",
    "job_description",
    "benefits",
    "skills",
    "responsibilities",
    "company_name",
    "company_profile",
]

for col in TEXT_COLUMNS:
    demo_jobs[col] = demo_jobs[col].apply(clean_text)

display(demo_jobs.head(3))

,job_id,experience,qualifications,location,country,work_type,job_posting_date,job_title,role,job_portal,job_description,benefits,skills,responsibilities,company_name,company_profile
0,1089843540111562,5 to 15 Years,M.Tech,Douglas,Isle of Man,Intern,2022-04-24,Digital Marketing Specialist,Social Media Manager,Snagajob,"Social Media Managers oversee an organizations social media presence. They create and schedule content, engage with followers, and analyze social media metrics to drive brand awareness and engagement.","{'Flexible Spending Accounts (FSAs), Relocation Assistance, Legal Assistance, Employee Recognition Programs, Financial Counseling'}","Social media platforms (e.g., Facebook, Twitter, Instagram) Content creation and scheduling Social media analytics and insights Community engagement Paid social advertising","Manage and grow social media accounts, create engaging content, and interact with the online community. Develop social media content calendars and strategies. Monitor social media trends and engagement metrics.",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversified Financials"",""City"":""Sunny Isles Beach"",""State"":""Florida"",""Zip"":""33160"",""Website"":""www.ielp.com"",""Ticker"":""IEP"",""CEO"":""David Willetts""}"
1,398454096642776,2 to 12 Years,BCA,Ashgabat,Turkmenistan,Intern,2022-12-19,Web Developer,Frontend Web Developer,Idealist,"Frontend Web Developers design and implement user interfaces for websites, ensuring they are visually appealing and user-friendly. They collaborate with designers and backend developers to create seamless web experiences for users.","{'Health Insurance, Retirement Plans, Paid Time Off (PTO), Flexible Work Arrangements, Employee Assistance Programs (EAP)'}","HTML, CSS, JavaScript Frontend frameworks (e.g., React, Angular) User experience (UX)","Design and code user interfaces for websites, ensuring a seamless and visually appealing user experience. Collaborate with UX designers to optimize user journeys. Ensure cross-browser compatibility and responsive design.",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Commercial Banks"",""City"":""Pittsburgh"",""State"":""Pennsylvania"",""Zip"":""15222"",""Website"":""www.pnc.com"",""Ticker"":""PNC"",""CEO"":""William S. Demchak""}"
2,481640072963533,0 to 12 Years,PhD,Macao,"Macao SAR, China",Temporary,2022-09-14,Operations Manager,Quality Control Manager,Jobs2Careers,"Quality Control Managers establish and enforce quality standards within an organization. They develop quality control processes, perform inspections, and implement corrective actions to maintain product or service quality.","{'Legal Assistance, Bonuses and Incentive Programs, Wellness Programs, Employee Discounts, Retirement Plans'}","Quality control processes and methodologies Statistical process control (SPC) Root cause analysis and corrective action Quality management systems (e.g., ISO 9001) Compliance and regulatory knowledge",Establish and enforce quality control standards and procedures. Conduct quality audits and inspections. Collaborate with production teams to address quality issues and implement improvements.,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: Property and Casualty (Stock)"",""City"":""San Antonio"",""State"":""Texas"",""Zip"":""78288"",""Website"":""www.usaa.com"",""Ticker"":"""",""CEO"":""Wayne Peacock""}"


In [9]:
# -----------------------------
# Build simple quality signals so we can remove weak rows.
#
# We want jobs that are useful for:
# - semantic matching
# - dashboard display
# - job details display
# -----------------------------
demo_jobs["description_len"] = demo_jobs["job_description"].str.len()
demo_jobs["skills_len"] = demo_jobs["skills"].str.len()
demo_jobs["responsibilities_len"] = demo_jobs["responsibilities"].str.len()

# Simple combined richness score
demo_jobs["quality_score"] = (
    demo_jobs["description_len"].fillna(0)
    + demo_jobs["skills_len"].fillna(0)
    + demo_jobs["responsibilities_len"].fillna(0)
)

display(
    demo_jobs[[
        "job_title",
        "company_name",
        "location",
        "description_len",
        "skills_len",
        "responsibilities_len",
        "quality_score"
    ]].head(5)
)

,job_title,company_name,location,description_len,skills_len,responsibilities_len,quality_score
0,Digital Marketing Specialist,Icahn Enterprises,Douglas,200,172,210,582
1,Web Developer,PNC Financial Services Group,Ashgabat,231,85,220,536
2,Operations Manager,United Services Automobile Assn.,Macao,222,199,191,612
3,Network Engineer,Hess,Porto-Novo,200,185,186,571
4,Event Manager,Cairn Energy,Santiago,235,114,158,507


In [10]:
# -----------------------------
# Keeping only rows that are strong enough for the demo.
#
# Rules:
# - must have title
# - must have company
# - must have location
# - must have non-empty description
# - description should be reasonably informative
# - should have some skills or responsibilities text
# -----------------------------
filtered_jobs = demo_jobs[
    (demo_jobs["job_title"] != "") &
    (demo_jobs["company_name"] != "") &
    (demo_jobs["location"] != "") &
    (demo_jobs["job_description"] != "") &
    (demo_jobs["description_len"] >= 200) &
    (
        (demo_jobs["skills_len"] >= 20) |
        (demo_jobs["responsibilities_len"] >= 40)
    )
].copy()

print("Original rows:", len(demo_jobs))
print("After quality filtering:", len(filtered_jobs))

display(filtered_jobs.head(3))

Original rows: 1615940
After quality filtering: 549156


,job_id,experience,qualifications,location,country,work_type,job_posting_date,job_title,role,job_portal,job_description,benefits,skills,responsibilities,company_name,company_profile,description_len,skills_len,responsibilities_len,quality_score
0,1089843540111562,5 to 15 Years,M.Tech,Douglas,Isle of Man,Intern,2022-04-24,Digital Marketing Specialist,Social Media Manager,Snagajob,"Social Media Managers oversee an organizations social media presence. They create and schedule content, engage with followers, and analyze social media metrics to drive brand awareness and engagement.","{'Flexible Spending Accounts (FSAs), Relocation Assistance, Legal Assistance, Employee Recognition Programs, Financial Counseling'}","Social media platforms (e.g., Facebook, Twitter, Instagram) Content creation and scheduling Social media analytics and insights Community engagement Paid social advertising","Manage and grow social media accounts, create engaging content, and interact with the online community. Develop social media content calendars and strategies. Monitor social media trends and engagement metrics.",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversified Financials"",""City"":""Sunny Isles Beach"",""State"":""Florida"",""Zip"":""33160"",""Website"":""www.ielp.com"",""Ticker"":""IEP"",""CEO"":""David Willetts""}",200,172,210,582
1,398454096642776,2 to 12 Years,BCA,Ashgabat,Turkmenistan,Intern,2022-12-19,Web Developer,Frontend Web Developer,Idealist,"Frontend Web Developers design and implement user interfaces for websites, ensuring they are visually appealing and user-friendly. They collaborate with designers and backend developers to create seamless web experiences for users.","{'Health Insurance, Retirement Plans, Paid Time Off (PTO), Flexible Work Arrangements, Employee Assistance Programs (EAP)'}","HTML, CSS, JavaScript Frontend frameworks (e.g., React, Angular) User experience (UX)","Design and code user interfaces for websites, ensuring a seamless and visually appealing user experience. Collaborate with UX designers to optimize user journeys. Ensure cross-browser compatibility and responsive design.",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Commercial Banks"",""City"":""Pittsburgh"",""State"":""Pennsylvania"",""Zip"":""15222"",""Website"":""www.pnc.com"",""Ticker"":""PNC"",""CEO"":""William S. Demchak""}",231,85,220,536
2,481640072963533,0 to 12 Years,PhD,Macao,"Macao SAR, China",Temporary,2022-09-14,Operations Manager,Quality Control Manager,Jobs2Careers,"Quality Control Managers establish and enforce quality standards within an organization. They develop quality control processes, perform inspections, and implement corrective actions to maintain product or service quality.","{'Legal Assistance, Bonuses and Incentive Programs, Wellness Programs, Employee Discounts, Retirement Plans'}","Quality control processes and methodologies Statistical process control (SPC) Root cause analysis and corrective action Quality management systems (e.g., ISO 9001) Compliance and regulatory knowledge",Establish and enforce quality control standards and procedures. Conduct quality audits and inspections. Collaborate with production teams to address quality issues and implement improvements.,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: Property and Casualty (Stock)"",""City"":""San Antonio"",""State"":""Texas"",""Zip"":""78288"",""Website"":""www.usaa.com"",""Ticker"":"""",""CEO"":""Wayne Peacock""}",222,199,191,612


In [11]:
# -----------------------------
# Removing obvious duplicates.
#
# Duplicate key built from:
# - title
# - company
# - location
# - first part of description
#
# This is a simple first-pass deduplication strategy.
# -----------------------------
filtered_jobs["dup_key"] = (
    filtered_jobs["job_title"].str.lower() + " | " +
    filtered_jobs["company_name"].str.lower() + " | " +
    filtered_jobs["location"].str.lower() + " | " +
    filtered_jobs["job_description"].str.lower().str[:200]
)

before_dedup = len(filtered_jobs)
filtered_jobs = filtered_jobs.drop_duplicates(subset=["dup_key"]).copy()
after_dedup = len(filtered_jobs)

print("Before dedup:", before_dedup)
print("After dedup:", after_dedup)

Before dedup: 549156
After dedup: 541207


In [12]:
# -----------------------------
# Inspect diversity before balanced sampling.
# This helps us keep a broad demo corpus rather than a narrow one.
# -----------------------------
print("Top roles:")
display(filtered_jobs["role"].value_counts().head(20))

print("\nTop work types:")
display(filtered_jobs["work_type"].value_counts().head(10))

print("\nTop countries:")
display(filtered_jobs["country"].value_counts().head(10))

Top roles:


role
User Interface Designer          13553
Social Media Manager             13446
User Experience Designer         13441
Social Media Analyst             10371
SEO Specialist                   10232
Backend Developer                10155
Office Manager                   10087
Customer Success Manager         10053
Frontend Developer               10024
Account Executive                 6924
Inside Sales Representative       6905
Personal Assistant                6901
Training Coordinator              6900
Data Scientist                    6861
IT Project Manager                6850
Business Intelligence Analyst     6835
Product Marketing Manager         6820
DevOps Engineer                   6812
Supply Chain Manager              6802
Benefits Coordinator              6734
Name: count, dtype: int64


Top work types:


work_type
Temporary    108667
Part-Time    108638
Contract     108081
Intern       108071
Full-Time    107750
Name: count, dtype: int64


Top countries:


country
Malta                             2635
Gambia                            2634
Sweden                            2624
St. Vincent and the Grenadines    2616
Fiji                              2600
Cuba                              2598
Guinea                            2592
Armenia                           2589
Turks and Caicos Islands          2586
Nepal                             2584
Name: count, dtype: int64

In [13]:
# -----------------------------
# Build a balanced subset by role.
#
# Strategy:
# - group by role
# - sample up to MAX_PER_ROLE jobs per role
#
# Some pandas versions may not preserve the grouping column
# reliably after groupby/apply, so we explicitly restore a
# fallback role column if needed.
# -----------------------------
MAX_PER_ROLE = 30

balanced_subset = (
    filtered_jobs
    .groupby("role", group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), MAX_PER_ROLE), random_state=42))
    .reset_index(drop=True)
)

# Defensive fallback:
# If pandas dropped the role column during groupby/apply,
# recreate it from job_title.
if "role" not in balanced_subset.columns:
    balanced_subset["role"] = balanced_subset["job_title"]

# Also fill any blank role values with job_title
balanced_subset["role"] = balanced_subset["role"].replace("", np.nan)
balanced_subset["role"] = balanced_subset["role"].fillna(balanced_subset["job_title"])

print("Balanced subset size:", len(balanced_subset))
print("Balanced subset columns:")
print(balanced_subset.columns.tolist())

display(balanced_subset.head(5))

Balanced subset size: 3720
Balanced subset columns:
['job_id', 'experience', 'qualifications', 'location', 'country', 'work_type', 'job_posting_date', 'job_title', 'job_portal', 'job_description', 'benefits', 'skills', 'responsibilities', 'company_name', 'company_profile', 'description_len', 'skills_len', 'responsibilities_len', 'quality_score', 'dup_key', 'role']


,job_id,experience,qualifications,location,country,work_type,job_posting_date,job_title,job_portal,job_description,benefits,skills,responsibilities,company_name,company_profile,description_len,skills_len,responsibilities_len,quality_score,dup_key,role
0,692765415301241,2 to 11 Years,B.Com,Minsk,Belarus,Contract,2023-05-13,Sales Representative,LinkedIn,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Tuition Reimbursement, Stock Options or Equity Grants, Parental Leave, Wellness Programs, Childcare Assistance'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",VMware,"{""Sector"":""Software"",""Industry"":""Computer Software"",""City"":""Palo Alto"",""State"":""California"",""Zip"":""94304"",""Website"":""www.vmware.com"",""Ticker"":""VMW"",""CEO"":""Raghu Raghuram""}",249,163,217,629,"sales representative | vmware | minsk | account executives manage and grow relationships with existing clients or customers. they understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often worki",Sales Representative
1,370087477853353,0 to 12 Years,B.Com,Berlin,Germany,Temporary,2023-05-31,Sales Representative,Glassdoor,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Employee Assistance Programs (EAP), Tuition Reimbursement, Profit-Sharing, Transportation Benefits, Parental Leave'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",Continental AG,"{""Sector"":""Automotive"",""Industry"":""Automotive"",""City"":""Hanover"",""State"":""N/A"",""Zip"":""N/A"",""Website"":""www.continental-corporation.com"",""Ticker"":""CON"",""CEO"":""Nikolai Setzer""}",249,163,217,629,"sales representative | continental ag | berlin | account executives manage and grow relationships with existing clients or customers. they understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often worki",Sales Representative
2,862414233805803,4 to 9 Years,BA,Saint George's,Grenada,Temporary,2022-06-25,Sales Representative,Jobs2Careers,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Transportation Benefits, Professional Development, Bonuses and Incentive Programs, Profit-Sharing, Employee Discounts'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",IBM (International Business Machines Corporation),"{""Sector"":""Technology/IT Services"",""Industry"":""Technology"",""City"":""Armonk"",""State"":""NY"",

In [14]:
# -----------------------------
# Applying a final cap only if needed.
# -----------------------------
TARGET_SIZE = 5000

if len(balanced_subset) > TARGET_SIZE:
    demo_subset = balanced_subset.sample(n=TARGET_SIZE, random_state=42).copy()
else:
    demo_subset = balanced_subset.copy()

# Defensive fallback again after final sampling
if "role" not in demo_subset.columns:
    demo_subset["role"] = demo_subset["job_title"]

demo_subset["role"] = demo_subset["role"].replace("", np.nan)
demo_subset["role"] = demo_subset["role"].fillna(demo_subset["job_title"])

print("Final demo subset size:", len(demo_subset))
print("demo_subset columns:")
print(demo_subset.columns.tolist())

Final demo subset size: 3720
demo_subset columns:
['job_id', 'experience', 'qualifications', 'location', 'country', 'work_type', 'job_posting_date', 'job_title', 'job_portal', 'job_description', 'benefits', 'skills', 'responsibilities', 'company_name', 'company_profile', 'description_len', 'skills_len', 'responsibilities_len', 'quality_score', 'dup_key', 'role']


In [15]:
# -----------------------------
# Keeping only the columns needed for downstream processing.
# -----------------------------
FINAL_SUBSET_COLUMNS = [
    "job_id",
    "experience",
    "qualifications",
    "location",
    "country",
    "work_type",
    "job_posting_date",
    "job_title",
    "role",
    "job_portal",
    "job_description",
    "benefits",
    "skills",
    "responsibilities",
    "company_name",
    "company_profile",
]

# Final safety check before column selection
if "role" not in demo_subset.columns:
    demo_subset["role"] = demo_subset["job_title"]

demo_subset = demo_subset[FINAL_SUBSET_COLUMNS].copy()

print("Final saved subset columns:")
print(demo_subset.columns.tolist())

display(demo_subset.head(3))

Final saved subset columns:
['job_id', 'experience', 'qualifications', 'location', 'country', 'work_type', 'job_posting_date', 'job_title', 'role', 'job_portal', 'job_description', 'benefits', 'skills', 'responsibilities', 'company_name', 'company_profile']


,job_id,experience,qualifications,location,country,work_type,job_posting_date,job_title,role,job_portal,job_description,benefits,skills,responsibilities,company_name,company_profile
0,692765415301241,2 to 11 Years,B.Com,Minsk,Belarus,Contract,2023-05-13,Sales Representative,Sales Representative,LinkedIn,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Tuition Reimbursement, Stock Options or Equity Grants, Parental Leave, Wellness Programs, Childcare Assistance'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",VMware,"{""Sector"":""Software"",""Industry"":""Computer Software"",""City"":""Palo Alto"",""State"":""California"",""Zip"":""94304"",""Website"":""www.vmware.com"",""Ticker"":""VMW"",""CEO"":""Raghu Raghuram""}"
1,370087477853353,0 to 12 Years,B.Com,Berlin,Germany,Temporary,2023-05-31,Sales Representative,Sales Representative,Glassdoor,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Employee Assistance Programs (EAP), Tuition Reimbursement, Profit-Sharing, Transportation Benefits, Parental Leave'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",Continental AG,"{""Sector"":""Automotive"",""Industry"":""Automotive"",""City"":""Hanover"",""State"":""N/A"",""Zip"":""N/A"",""Website"":""www.continental-corporation.com"",""Ticker"":""CON"",""CEO"":""Nikolai Setzer""}"
2,862414233805803,4 to 9 Years,BA,Saint George's,Grenada,Temporary,2022-06-25,Sales Representative,Sales Representative,Jobs2Careers,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Transportation Benefits, Professional Development, Bonuses and Incentive Programs, Profit-Sharing, Employee Discounts'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",IBM (International Business Machines Corporation),"{""Sector"":""Technology/IT Services"",""Industry"":""Technology"",""City"":""Armonk"",""State"":""NY"",""Zip"":""10504"",""Website"":""https://www.ibm.com/"",""Ticker"":""IBM"",""CEO"":""Arvind Krishna""}"


In [16]:
# -----------------------------
# Save the reduced subset as an intermediate file.
# This becomes the input to Notebook 09.
# -----------------------------
DEMO_SUBSET_PARQUET = DATA_PROCESSED / "demo_jobs_subset.parquet"

demo_subset.to_parquet(DEMO_SUBSET_PARQUET, index=False)

print("Saved reduced subset to:", DEMO_SUBSET_PARQUET)

Saved reduced subset to: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\processed\demo_jobs_subset.parquet


In [17]:
# -----------------------------
# Reloading the saved parquet and verifying that the schema is correct.
# -----------------------------
check_subset = pd.read_parquet(DEMO_SUBSET_PARQUET)

print("Saved parquet shape:", check_subset.shape)
print("Saved parquet columns:")
print(check_subset.columns.tolist())

display(check_subset.head(3))

Saved parquet shape: (3720, 16)
Saved parquet columns:
['job_id', 'experience', 'qualifications', 'location', 'country', 'work_type', 'job_posting_date', 'job_title', 'role', 'job_portal', 'job_description', 'benefits', 'skills', 'responsibilities', 'company_name', 'company_profile']


,job_id,experience,qualifications,location,country,work_type,job_posting_date,job_title,role,job_portal,job_description,benefits,skills,responsibilities,company_name,company_profile
0,692765415301241,2 to 11 Years,B.Com,Minsk,Belarus,Contract,2023-05-13,Sales Representative,Sales Representative,LinkedIn,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Tuition Reimbursement, Stock Options or Equity Grants, Parental Leave, Wellness Programs, Childcare Assistance'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",VMware,"{""Sector"":""Software"",""Industry"":""Computer Software"",""City"":""Palo Alto"",""State"":""California"",""Zip"":""94304"",""Website"":""www.vmware.com"",""Ticker"":""VMW"",""CEO"":""Raghu Raghuram""}"
1,370087477853353,0 to 12 Years,B.Com,Berlin,Germany,Temporary,2023-05-31,Sales Representative,Sales Representative,Glassdoor,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Employee Assistance Programs (EAP), Tuition Reimbursement, Profit-Sharing, Transportation Benefits, Parental Leave'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",Continental AG,"{""Sector"":""Automotive"",""Industry"":""Automotive"",""City"":""Hanover"",""State"":""N/A"",""Zip"":""N/A"",""Website"":""www.continental-corporation.com"",""Ticker"":""CON"",""CEO"":""Nikolai Setzer""}"
2,862414233805803,4 to 9 Years,BA,Saint George's,Grenada,Temporary,2022-06-25,Sales Representative,Sales Representative,Jobs2Careers,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Transportation Benefits, Professional Development, Bonuses and Incentive Programs, Profit-Sharing, Employee Discounts'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",IBM (International Business Machines Corporation),"{""Sector"":""Technology/IT Services"",""Industry"":""Technology"",""City"":""Armonk"",""State"":""NY"",""Zip"":""10504"",""Website"":""https://www.ibm.com/"",""Ticker"":""IBM"",""CEO"":""Arvind Krishna""}"


## Demo Subset Creation Summary

Because the raw Kaggle job postings file is very large, the full dataset is not used directly in the demo pipeline. Instead, a reduced demo subset is created by:

- loading only the columns needed for recommendation and UI display
- handling column-name inconsistencies safely
- ensuring a stable internal schema, including a fallback `role` field
- removing rows with weak or incomplete job information
- removing obvious duplicates
- sampling across job roles to preserve diversity
- capping the final size to a manageable demo corpus

This produces a smaller, higher-quality, and schema-stable job set that is more suitable for embedding, recommendation, and user-facing presentation.t.head(3))sentation.